# Análisis de acciones tecnológicas: Apple, Microsoft y NVIDIA

**Autora:** Tamara Parra · Analista de Datos

**Pregunta:** si alguien hubiera invertido USD 1.000 en cada una de estas tres empresas hace un año, ¿cuál habría rendido más y cuál fue más inestable?

**Datos:** precios de cierre diarios descargados de Yahoo Finance con la librería `yfinance`, entre el 17 de septiembre de 2025 y el 16 de septiembre de 2026.

**Pasos:**
1. Descarga y revisión de los datos
2. Estadísticas descriptivas
3. Evolución de precios
4. Simulación de una inversión de USD 1.000
5. Rentabilidad, volatilidad y caída máxima
6. Relación entre las tres acciones
7. Conclusiones

> Este es un ejercicio de análisis de datos. No es una recomendación de inversión.

## 0. Librerías

In [ ]:
# Si falta alguna librería, descomenta la línea siguiente y ejecútala una vez
#%pip install --upgrade "kaleido>=1"
import pandas as pd
import yfinance as yf
import plotly.express as px
import plotly.io as pio

pio.templates.default = "plotly_white"
COLORES = {"AAPL": "#0369a1", "MSFT": "#6d4fd8", "NVDA": "#b45309"}
NOMBRES = {"AAPL": "Apple", "MSFT": "Microsoft", "NVDA": "NVIDIA"}

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Descarga y revisión de los datos

Uso fechas fijas para que el análisis se pueda repetir y dé los mismos resultados.

In [2]:
tickers = ["AAPL", "MSFT", "NVDA"]

datos = yf.download(tickers, start="2025-09-17", end="2026-09-17", progress=False)
precios = datos["Close"].dropna()

print(f"Días hábiles con datos: {len(precios)}")
print(f"Desde {precios.index.min():%d-%m-%Y} hasta {precios.index.max():%d-%m-%Y}")
precios.head()

Días hábiles con datos: 251
Desde 17-09-2025 hasta 16-09-2026


Ticker,AAPL,MSFT,NVDA
Date,,,
2025-09-17,238.110443,505.857971,169.882980
2025-09-18,237.004501,504.300812,175.818787
2025-09-19,244.596466,513.703369,176.247742
2025-09-22,255.137512,510.251770,183.171158
2025-09-23,253.493591,505.074463,178.003540


In [3]:
# Revisión de calidad: valores faltantes por empresa
precios.isna().sum()

Ticker
AAPL    0
MSFT    0
NVDA    0
dtype: int64

## 2. Estadísticas descriptivas

In [4]:
precios.describe().round(2)

Ticker,AAPL,MSFT,NVDA
count,251.00,251.00,251.00
mean,280.32,445.59,195.25
std,25.42,51.01,15.70
min,237.00,352.17,164.79
25%,259.48,398.70,182.72
50%,272.66,449.39,190.09
75%,302.70,492.16,208.10
max,339.79,537.65,235.20


## 3. Evolución de precios

In [5]:
fig = px.line(
    precios.rename(columns=NOMBRES),
    title="Precio de cierre diario (USD), últimos 12 meses",
    labels={"value": "Precio (USD)", "Date": "Fecha", "Ticker": "Empresa"},
    color_discrete_sequence=list(COLORES.values()),
)
fig.show()

In [6]:
promedio = precios.mean().rename(index=NOMBRES)

fig = px.bar(
    promedio,
    title="Precio promedio en el período (USD)",
    labels={"value": "Precio promedio (USD)", "Ticker": "Empresa"},
    color=promedio.index,
    color_discrete_sequence=list(COLORES.values()),
    text_auto=".0f",
)
fig.update_layout(showlegend=False)
fig.show()

El precio promedio no sirve para comparar rendimientos: Microsoft cuesta más por acción, pero eso no significa que haya ganado más. Para comparar hay que llevar todo a la misma base.

## 4. ¿Cuánto tendrías hoy si hubieras invertido USD 1.000?

In [7]:
inversion = precios / precios.iloc[0] * 1000

fig = px.line(
    inversion.rename(columns=NOMBRES),
    title="Valor de una inversión de USD 1.000 en cada empresa",
    labels={"value": "Valor (USD)", "Date": "Fecha", "Ticker": "Empresa"},
    color_discrete_sequence=list(COLORES.values()),
)
fig.add_hline(y=1000, line_dash="dot", line_color="gray")
fig.show()

inversion.iloc[-1].rename(index=NOMBRES).round(0)

Ticker
Apple        1396.0
Microsoft     969.0
NVIDIA       1259.0
Name: 2026-09-16 00:00:00, dtype: float64

## 5. Rentabilidad, volatilidad y caída máxima

In [8]:
retornos = precios.pct_change().dropna()

resumen = pd.DataFrame({
    "Rentabilidad del período (%)": (precios.iloc[-1] / precios.iloc[0] - 1) * 100,
    "Volatilidad anual (%)": retornos.std() * (252 ** 0.5) * 100,
    "Mejor día (%)": retornos.max() * 100,
    "Peor día (%)": retornos.min() * 100,
    "Caída máxima (%)": ((precios / precios.cummax()) - 1).min() * 100,
}).rename(index=NOMBRES).round(1)

resumen

,Rentabilidad del período (%),Volatilidad anual (%),Mejor día (%),Peor día (%),Caída máxima (%)
Ticker,,,,,
Apple,39.6,25.1,4.8,-7.4,-13.8
Microsoft,-3.1,32.6,15.5,-10.0,-34.5
NVIDIA,25.9,38.1,8.7,-6.2,-20.2


- **Rentabilidad del período:** cuánto cambió el precio entre el primer y el último día.
- **Volatilidad anual:** cuánto se mueve el precio día a día, llevado a escala anual. A mayor volatilidad, más riesgo.
- **Caída máxima:** la peor baja desde un máximo anterior. Muestra cuánto habría tenido que aguantar alguien en el peor momento.

## 6. Relación entre las tres acciones

In [9]:
correlacion = retornos.rename(columns=NOMBRES).corr().round(2)

fig = px.imshow(
    correlacion,
    text_auto=True,
    color_continuous_scale="Blues",
    zmin=0, zmax=1,
    title="Correlación de los retornos diarios",
)
fig.show()

Una correlación cercana a 1 indica que las acciones suelen subir y bajar juntas. Si es baja, combinar las acciones reduce el riesgo total.

## 7. Conclusiones

- **Apple fue la que más rindió:** USD 1.000 invertidos el 17-09-2025 valían cerca de **USD 1.396** el 16-09-2026 (+39,6 %).
- **NVIDIA quedó en segundo lugar:** cerca de **USD 1.259** (+25,9 %).
- **Microsoft terminó levemente abajo:** cerca de **USD 969** (−3,1 %), aunque tiene el precio por acción más alto. Por eso el precio promedio no sirve para comparar rendimientos.
- Para decidir no basta con la rentabilidad: la tabla del punto 5 muestra cuánto riesgo (volatilidad y caída máxima) hubo detrás de cada resultado.

**Próximos pasos:** comparar contra un índice de referencia como el S&P 500 y ampliar el período a 3 o 5 años.

## Anexo: exportar los gráficos

In [ ]:
import os
os.makedirs("graficos", exist_ok=True)

fig_inv = px.line(
    inversion.rename(columns=NOMBRES),
    title="Valor de una inversión de USD 1.000 en cada empresa",
    labels={"value": "Valor (USD)", "Date": "Fecha", "Ticker": "Empresa"},
    color_discrete_sequence=list(COLORES.values()),
)
fig_inv.write_html("graficos/inversion_1000_usd.html")  # versión interactiva

try:
    fig_inv.write_image("graficos/inversion_1000_usd.png", width=1100, height=550, scale=2)  # imagen para el README
    print("Gráficos guardados en la carpeta graficos/")
except Exception as e:
    print("No se pudo crear el PNG (requiere el paquete kaleido):", e)

Gráficos guardados en la carpeta graficos/
